# NumPy for Quantum Mathematics



We will cover seven topics in this notebook, in the same order as the mathematics notebook.

1. Complex numbers in NumPy
2. Vectors and kets
3. Inner products
4. Matrices
5. Eigenvalues and eigenvectors
6. Tensor products
7. Checking if a matrix is unitary


## 1. Complex Numbers in NumPy

Recall that a complex number is written as

$$z = a + bi$$

where $a$ is the real part and $b$ is the imaginary part.

In Python, the imaginary unit is written as `j` instead of `i`. This is a convention borrowed from electrical engineering, where `i` is already used for current. Do not let this confuse you, `j` in NumPy plays exactly the role that $i$ plays in the mathematics notebook.

In [1]:
z = 3 + 4j

print(z)
print(type(z))

(3+4j)
<class 'complex'>


NumPy gives us functions to extract the real part, the imaginary part, the modulus, and the conjugate of a complex number. These correspond directly to the formulas

$$\text{Re}(z) = a, \qquad \text{Im}(z) = b, \qquad |z| = \sqrt{a^2 + b^2}, \qquad z^* = a - bi$$

In [2]:
import numpy as np

z = 3 + 4j

real_part = z.real
imaginary_part = z.imag
modulus = np.abs(z)
conjugate = np.conj(z)

print("Real part:", real_part)
print("Imaginary part:", imaginary_part)
print("Modulus:", modulus)
print("Conjugate:", conjugate)

Real part: 3.0
Imaginary part: 4.0
Modulus: 5.0
Conjugate: (3-4j)


Notice that the modulus matches the result from the mathematics notebook, where we computed $|3 + 4i| = 5$ by hand.

We can also verify the identity

$$|z|^2 = z z^*$$

directly in code.

In [3]:
z = 3 + 4j

left_side = np.abs(z) ** 2
right_side = z * np.conj(z)

print("Left side, modulus squared:", left_side)
print("Right side, z times z conjugate:", right_side)

Left side, modulus squared: 25.0
Right side, z times z conjugate: (25+0j)


The two values match, confirming the identity numerically. You will notice that the right side prints as a complex number with a zero imaginary part, such as `(25+0j)`. This is expected, NumPy keeps the result in complex form even when the imaginary part vanishes.

### Worked example

Euler's formula states

$$e^{i\theta} = \cos\theta + i\sin\theta$$

We can verify this for a specific angle, say $\theta = \pi$, which gives Euler's identity $e^{i\pi} + 1 = 0$.

In [4]:
theta = np.pi

left_side = np.exp(1j * theta)
right_side = np.cos(theta) + 1j * np.sin(theta)

print("e to the i pi:", left_side)
print("cos(pi) + i sin(pi):", right_side)

e to the i pi: (-1+1.2246467991473532e-16j)
cos(pi) + i sin(pi): (-1+1.2246467991473532e-16j)


The output will be very close to negative one, but not exactly negative one, due to the small rounding errors that are unavoidable when a computer works with decimal approximations of numbers like $\pi$. This kind of tiny numerical error appears constantly in scientific computing and is nothing to worry about.

## 2. Vectors and Kets

A vector in the mathematics notebook was written as a column,

$$v = \begin{pmatrix} v_1 \\ v_2 \end{pmatrix}$$

In quantum mechanics, this same column vector is often written using a special notation called a ket, written as $|v\rangle$. The two notations describe exactly the same mathematical object, a ket is simply a column vector, written in a way that makes it visually clear we are talking about a quantum state.

In NumPy, we represent a vector using a one dimensional array.

In [5]:
v = np.array([2, 3])
u = np.array([4, 1])

print(v)
print(u)

[2 3]
[4 1]


For quantum states, the entries are usually complex numbers. The two most important kets in quantum computing are

$$|0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

These represent the two basic states of a single qubit, analogous to the classical bit values 0 and 1.

In [6]:
ket_zero = np.array([1, 0])
ket_one = np.array([0, 1])

print("Ket zero:", ket_zero)
print("Ket one:", ket_one)

Ket zero: [1 0]
Ket one: [0 1]


Vector addition and scalar multiplication work exactly the way you expect from the mathematics notebook.

In [7]:
u = np.array([2, 5])
v = np.array([4, 1])

vector_sum = u + v
scaled_vector = 3 * u

print("u + v:", vector_sum)
print("3 times u:", scaled_vector)

u + v: [6 6]
3 times u: [ 6 15]


A very important quantum state is the superposition state

$$|+\rangle = \frac{1}{\sqrt{2}} \big(|0\rangle + |1\rangle\big)$$

This represents a qubit that is, in a sense, equally in the state $|0\rangle$ and the state $|1\rangle$ at the same time. We will build this state directly using NumPy.

In [8]:
ket_zero = np.array([1, 0])
ket_one = np.array([0, 1])

normalization_factor = 1 / np.sqrt(2)
ket_plus = normalization_factor * (ket_zero + ket_one)

print("Ket plus:", ket_plus)

Ket plus: [0.70710678 0.70710678]


### Worked example, the norm of a vector

Recall that the norm of a vector is

$$\|v\| = \sqrt{v_1^2 + v_2^2}$$

NumPy provides a built in function for this, `np.linalg.norm`.

In [9]:
v = np.array([3, 4])

norm_of_v = np.linalg.norm(v)
print("Norm of v:", norm_of_v)

Norm of v: 5.0


This matches the hand calculation from the mathematics notebook, where $\|(3,4)\| = 5$.

We can also check that the state $|+\rangle$ we built above has norm equal to one, which it must, since every valid quantum state has unit norm.

In [10]:
norm_of_plus = np.linalg.norm(ket_plus)
print("Norm of ket plus:", norm_of_plus)

Norm of ket plus: 0.9999999999999999


## 3. Inner Products

For complex vectors, recall that the inner product is

$$\langle u, v \rangle = u^{\dagger} v$$

where $u^{\dagger}$ is the conjugate transpose of $u$. In NumPy, the function `np.vdot` computes exactly this, it automatically takes the complex conjugate of the first argument before multiplying.

In [11]:
u = np.array([1 + 1j, 2])
v = np.array([3, 1j])

inner_product = np.vdot(u, v)
print("Inner product of u and v:", inner_product)

Inner product of u and v: (3-1j)


This matches the hand calculation from the mathematics notebook, where we computed $\langle u, v \rangle = 3 - i$ for the same two vectors.

It is worth being careful here. NumPy also has a function called `np.dot`, but that function does not take the complex conjugate. For real vectors this makes no difference, but for complex vectors, always use `np.vdot` when computing a quantum mechanical inner product.

In [12]:
u = np.array([1 + 1j, 2])
v = np.array([3, 1j])

result_using_vdot = np.vdot(u, v)
result_using_dot = np.dot(u, v)

print("Using vdot, correct for quantum inner products:", result_using_vdot)
print("Using dot, this does not conjugate:", result_using_dot)

Using vdot, correct for quantum inner products: (3-1j)
Using dot, this does not conjugate: (3+5j)


### Worked example, checking orthogonality

Two vectors are orthogonal if their inner product is zero. The states $|0\rangle$ and $|1\rangle$ should be orthogonal, since they represent perfectly distinguishable measurement outcomes.

In [13]:
ket_zero = np.array([1, 0])
ket_one = np.array([0, 1])

inner_product = np.vdot(ket_zero, ket_one)
print("Inner product of ket zero and ket one:", inner_product)

Inner product of ket zero and ket one: 0


The result is zero, confirming that $|0\rangle$ and $|1\rangle$ form an orthogonal pair, exactly as stated in the mathematics notebook.

## 4. Matrices

A matrix in NumPy is represented as a two dimensional array. Recall a general matrix from the mathematics notebook,

$$A = \begin{pmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{pmatrix}$$

In [14]:
A = np.array([[1, 2],
              [3, 4]])

print(A)

[[1 2]
 [3 4]]


Matrix addition and scalar multiplication work the same way as for vectors.

In [15]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

matrix_sum = A + B
scaled_matrix = 3 * A

print("A + B:")
print(matrix_sum)
print()
print("3 times A:")
print(scaled_matrix)

A + B:
[[ 6  8]
 [10 12]]

3 times A:
[[ 3  6]
 [ 9 12]]


Matrix multiplication is different from element by element multiplication. In NumPy, the `*` symbol multiplies matrices element by element, which is **not** what we usually want. To perform true matrix multiplication, use `np.matmul`, or equivalently the `@` symbol.

In [16]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

elementwise_product = A * B
true_matrix_product = A @ B

print("Element by element, usually not what we want:")
print(elementwise_product)
print()
print("True matrix multiplication:")
print(true_matrix_product)

Element by element, usually not what we want:
[[ 5 12]
 [21 32]]

True matrix multiplication:
[[19 22]
 [43 50]]


We can also multiply a matrix by a vector, which is how a quantum gate acts on a quantum state.

In [17]:
A = np.array([[1, 2],
              [3, 4]])
v = np.array([5, 6])

result = A @ v
print(result)

[17 39]


This matches the hand calculation from the mathematics notebook.

Three operations on matrices will appear constantly in this course, the transpose, the complex conjugate, and the Hermitian transpose, which combines both.

In [18]:
A = np.array([[1 + 1j, 2],
              [3 - 1j, 4j]])

transpose_of_A = A.T
conjugate_of_A = np.conj(A)
hermitian_transpose_of_A = A.conj().T

print("Transpose:")
print(transpose_of_A)
print()
print("Complex conjugate:")
print(conjugate_of_A)
print()
print("Hermitian transpose, conjugate then transpose:")
print(hermitian_transpose_of_A)

Transpose:
[[1.+1.j 3.-1.j]
 [2.+0.j 0.+4.j]]

Complex conjugate:
[[1.-1.j 2.-0.j]
 [3.+1.j 0.-4.j]]

Hermitian transpose, conjugate then transpose:
[[1.-1.j 3.+1.j]
 [2.-0.j 0.-4.j]]


The Hermitian transpose, written $A^{\dagger}$ in the mathematics notebook, is the most important of the three for quantum computing. It appears in the definition of the inner product, in the test for Hermitian matrices, and in the test for unitary matrices.

### Worked example, the identity matrix

Recall that the identity matrix satisfies $AI = IA = A$.

In [19]:
identity_matrix = np.eye(2)
print(identity_matrix)

A = np.array([[1, 2],
              [3, 4]])

product_with_identity = A @ identity_matrix
print(product_with_identity)

[[1. 0.]
 [0. 1.]]
[[1. 2.]
 [3. 4.]]


## 5. Eigenvalues and Eigenvectors

An eigenvector of a matrix $A$ is a vector $v$ that, when multiplied by $A$, does not change direction, only its length is scaled. The scaling factor is called the eigenvalue.

This is captured by the equation

$$Av = \lambda v$$

where $\lambda$ is the eigenvalue and $v$ is the corresponding eigenvector.

NumPy computes eigenvalues and eigenvectors for us using `np.linalg.eig`.

In [20]:
A = np.array([[2, 0],
              [0, 3]])

eigenvalues, eigenvectors = np.linalg.eig(A)

print("Eigenvalues:", eigenvalues)
print("Eigenvectors:")
print(eigenvectors)

Eigenvalues: [2. 3.]
Eigenvectors:
[[1. 0.]
 [0. 1.]]


Each column of the eigenvectors array corresponds to one eigenvalue. For this diagonal matrix, the eigenvalues are simply the diagonal entries, and the eigenvectors are the standard basis vectors, which matches what we would expect by inspection.

We can verify the defining equation $Av = \lambda v$ directly.

In [21]:
A = np.array([[2, 0],
              [0, 3]])

eigenvalues, eigenvectors = np.linalg.eig(A)

first_eigenvalue = eigenvalues[0]
first_eigenvector = eigenvectors[:, 0]

left_side = A @ first_eigenvector
right_side = first_eigenvalue * first_eigenvector

print("A times the eigenvector:", left_side)
print("Eigenvalue times the eigenvector:", right_side)

A times the eigenvector: [2. 0.]
Eigenvalue times the eigenvector: [2. 0.]


The two sides match, confirming the eigenvector equation numerically.

### Worked example, a qubit gate

In quantum computing, the Pauli Z gate is represented by the matrix

$$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

This gate leaves $|0\rangle$ unchanged and flips the sign of $|1\rangle$. We expect the eigenvalues to be $+1$ and $-1$, with $|0\rangle$ and $|1\rangle$ as the corresponding eigenvectors.

In [22]:
pauli_z = np.array([[1, 0],
                     [0, -1]])

eigenvalues, eigenvectors = np.linalg.eig(pauli_z)

print("Eigenvalues of the Z gate:", eigenvalues)
print("Eigenvectors of the Z gate:")
print(eigenvectors)

Eigenvalues of the Z gate: [ 1. -1.]
Eigenvectors of the Z gate:
[[1. 0.]
 [0. 1.]]


The eigenvalues are exactly $+1$ and $-1$, as expected. This is not a coincidence, eigenvalues of quantum gates correspond directly to the possible measurement outcomes associated with that gate, and the eigenvectors are the quantum states that produce those outcomes with certainty.

## 6. Tensor Products

Recall the tensor product of two vectors,

$$u \otimes v = \begin{pmatrix} u_1 v_1 \\ u_1 v_2 \\ u_2 v_1 \\ u_2 v_2 \end{pmatrix}$$

This operation lets us combine two separate quantum systems, such as two qubits, into a single combined description. NumPy provides this operation through `np.kron`, short for the Kronecker product, which is the technical name for the tensor product when applied to arrays.

In [23]:
u = np.array([1, 2])
v = np.array([3, 4])

tensor_product = np.kron(u, v)
print(tensor_product)

[3 4 6 8]


This matches the hand calculation from the mathematics notebook exactly.

We can also compute the tensor product of two qubit states. For instance, $|0\rangle \otimes |1\rangle$ represents a two qubit system where the first qubit is in state $|0\rangle$ and the second qubit is in state $|1\rangle$.

In [24]:
ket_zero = np.array([1, 0])
ket_one = np.array([0, 1])

combined_state = np.kron(ket_zero, ket_one)
print(combined_state)

[0 1 0 0]


Notice that the resulting vector has four entries instead of two. In general, combining two qubits gives a vector space of dimension $2 \times 2 = 4$, combining three qubits gives dimension $2^3 = 8$, and so on. This exponential growth in dimension is one of the central reasons quantum computers are powerful, and also one of the central reasons they are hard to simulate classically.

The tensor product also works on matrices, which lets us describe a gate acting on one qubit while leaving another qubit untouched.

In [25]:
identity_matrix = np.eye(2)
pauli_z = np.array([[1, 0],
                     [0, -1]])

combined_gate = np.kron(identity_matrix, pauli_z)
print(combined_gate)

[[ 1.  0.  0.  0.]
 [ 0. -1.  0. -0.]
 [ 0.  0.  1.  0.]
 [ 0. -0.  0. -1.]]


This four by four matrix represents the Z gate acting only on the second qubit of a two qubit system, while the first qubit is left alone by the identity matrix.

### Worked example, building a two qubit basis state

We can construct $|1\rangle \otimes |0\rangle$ and verify it is a unit vector, as every valid quantum state must be.

In [26]:
ket_zero = np.array([1, 0])
ket_one = np.array([0, 1])

state_one_zero = np.kron(ket_one, ket_zero)
print("State:", state_one_zero)

norm_of_state = np.linalg.norm(state_one_zero)
print("Norm:", norm_of_state)

State: [0 0 1 0]
Norm: 1.0


## 7. Checking if a Matrix is Unitary

Recall from the mathematics notebook that a matrix $U$ is unitary if

$$U^{\dagger} U = I$$

This condition is essential in quantum computing because every quantum gate must be unitary. A unitary gate preserves the norm of a quantum state, which corresponds physically to the fact that probabilities must always add up to one, before and after applying a gate.

We will now write a small, reusable function that checks whether a given matrix is unitary. This function will be useful for the rest of the course whenever we want to confirm that a gate we have written down is physically valid.

In [27]:
def is_unitary(matrix):
    hermitian_transpose = matrix.conj().T
    product = hermitian_transpose @ matrix
    identity_matrix = np.eye(matrix.shape[0])
    return np.allclose(product, identity_matrix)

Notice the use of `np.allclose` instead of checking for exact equality. This is important. Due to small rounding errors in floating point arithmetic, the product $U^{\dagger}U$ will almost never be exactly equal to the identity matrix, it will only be extremely close. The function `np.allclose` checks that two arrays match within a small numerical tolerance, which is the correct way to compare floating point results.

Let us test this function on the Pauli Z gate, which we already know should be unitary.

In [28]:
pauli_z = np.array([[1, 0],
                     [0, -1]])

result = is_unitary(pauli_z)
print("Is the Z gate unitary:", result)

Is the Z gate unitary: True


Now let us test it on the Hadamard gate, one of the most important gates in quantum computing, defined as

$$H = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

In [29]:
normalization_factor = 1 / np.sqrt(2)
hadamard = normalization_factor * np.array([[1, 1],
                                             [1, -1]])

result = is_unitary(hadamard)
print("Is the Hadamard gate unitary:", result)

Is the Hadamard gate unitary: True


Finally, let us test the function on a matrix that should fail the check, to make sure our function correctly identifies non-unitary matrices as well.

In [30]:
not_unitary_matrix = np.array([[1, 2],
                                [3, 4]])

result = is_unitary(not_unitary_matrix)
print("Is this matrix unitary:", result)

Is this matrix unitary: False
